# Baseline Models

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import mlflow.xgboost
from pyspark.sql import functions as F
from sklearn.metrics import f1_score, accuracy_score, classification_report
from collections import Counter

#### 1. Load data

In [0]:
ml_features_df = spark.table(tables["ml_features"])

feature_cols = [c for c in ml_feature_columns if c in ml_features_df.columns]
print(f"Feature columns: {len(feature_cols)}")

_tmp_spark = "/tmp/wyckoff_ml_features_08"   # dbfs:/tmp/...
_tmp_local = "/dbfs/tmp/wyckoff_ml_features_08"  # pandas reads FUSE mount
dbutils.fs.rm(_tmp_spark, recurse=True)

for _split in ["train", "val", "test"]:
    ml_features_df.filter(F.col("split") == _split) \
        .write.mode("overwrite").parquet(f"{_tmp_spark}/{_split}")
    print(f"  Written {_split} parquet")

train_pdf = pd.read_parquet(f"{_tmp_local}/train").sort_values(["symbol", "date"]).reset_index(drop=True)
val_pdf = pd.read_parquet(f"{_tmp_local}/val").sort_values(["symbol", "date"]).reset_index(drop=True)
test_pdf = pd.read_parquet(f"{_tmp_local}/test").sort_values(["symbol", "date"]).reset_index(drop=True)

dbutils.fs.rm(_tmp_spark, recurse=True)
print(f"  Train: {len(train_pdf):,}  Val: {len(val_pdf):,}  Test: {len(test_pdf):,}")

In [0]:
def create_windows(pdf, feature_cols, window_size, stride, pl_threshold):
    """
    Create sliding windows from flat per-day data.
    Per-window z-score normalization. Filters out price-limit-heavy windows.
    Pre-allocates output in float32 to avoid 2x peak memory.
    """
    n_feat = len(feature_cols)

    # Upper-bound estimate for pre-allocation
    max_windows = sum(
        max(0, (len(g) - window_size) // stride + 1)
        for _, g in pdf.groupby("symbol", sort=False)
    )
    if max_windows == 0:
        return np.array([]), np.array([]), [], []

    features = np.empty((max_windows, window_size, n_feat), dtype=np.float32)
    labels = np.empty(max_windows, dtype=np.int64)
    symbols_out, dates_out = [], []
    idx = 0

    for sym, group in pdf.groupby("symbol", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        n = len(group)
        if n < window_size:
            continue

        feat = group[feature_cols].values.astype(np.float32)
        feat = np.where(np.isinf(feat), np.nan, feat)
        feat_df = pd.DataFrame(feat)
        feat = feat_df.ffill().fillna(0.0).values

        labs = group["phase_label"].values
        dts = group["date"].values
        pl = group["price_limit_hit"].values if "price_limit_hit" in group.columns else np.zeros(n)

        for i in range(window_size - 1, n, stride):
            s = i - window_size + 1

            if np.nansum(pl[s:i+1]) / window_size > pl_threshold:
                continue

            window = feat[s:i+1]

            mu = np.nanmean(window, axis=0, keepdims=True)
            sigma = np.nanstd(window, axis=0, keepdims=True)
            sigma = np.where(sigma == 0, 1.0, sigma)
            features[idx] = (window - mu) / sigma
            np.nan_to_num(features[idx], copy=False, nan=0.0, posinf=0.0, neginf=0.0)

            labels[idx] = int(labs[i])
            symbols_out.append(sym)
            dates_out.append(dts[i])
            idx += 1

    return features[:idx], labels[:idx], symbols_out, dates_out


print("Create sliding windows")
train_features, train_labels, _, _ = create_windows(
    train_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del train_pdf
print(f"  Train windows: {len(train_labels):,}")

val_features, val_labels, _, _ = create_windows(
    val_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del val_pdf
print(f"  Val windows:   {len(val_labels):,}")

test_features, test_labels, test_symbols, test_dates = create_windows(
    test_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del test_pdf
print(f"  Test windows:  {len(test_labels):,}")

if len(train_features) == 0:
    raise ValueError("Training set is empty - check that ml_features table has 'train' split data.")
if len(val_features) == 0:
    raise ValueError("Validation set is empty - check that ml_features table has 'val' split data.")

window_size, num_features = train_features.shape[1], train_features.shape[2]
print(f"\nTrain: {len(train_features):,}, Val: {len(val_features):,}, Test: {len(test_features):,}")
print(f"Window: {window_size} × {num_features}")

assert train_features.shape[1:] == val_features.shape[1:] == test_features.shape[1:], \
    f"Feature shape mismatch: train={train_features.shape[1:]}, val={val_features.shape[1:]}, test={test_features.shape[1:]}"

In [0]:
cfg = CNNLSTMConfig()

if cfg.augment_rare_classes and len(train_features) > 0:
    class_counts = Counter(train_labels.tolist())
    target_count = cfg.augment_max_target

    rare_classes = [cls for cls, cnt in class_counts.items() if cnt < target_count]
    print(f"Class distribution before augmentation:")
    for cls in sorted(class_counts):
        marker = " ← RARE (will augment)" if cls in rare_classes else ""
        print(f"  Class {cls}: {class_counts[cls]:,}{marker}")
    print(f"Augment target cap: {target_count:,}")

    aug_features_list = []
    aug_labels_list = []
    rng = np.random.RandomState(42)

    for cls in rare_classes:
        cls_mask = train_labels == cls
        cls_features = train_features[cls_mask]
        n_existing = len(cls_features)
        n_needed = target_count - n_existing
        if n_needed <= 0:
            continue

        indices = rng.choice(n_existing, size=n_needed, replace=True)
        sampled = cls_features[indices].copy()

        noise = rng.normal(0, cfg.augment_jitter_std, size=sampled.shape).astype(np.float32)
        sampled = sampled + noise

        aug_features_list.append(sampled)
        aug_labels_list.append(np.full(n_needed, cls, dtype=np.int64))
        print(f"  Augmented class {cls}: {n_existing:,} → {n_existing + n_needed:,} (+{n_needed:,})")

    if aug_features_list:
        train_features = np.concatenate([train_features] + aug_features_list, axis=0)
        train_labels = np.concatenate([train_labels] + aug_labels_list, axis=0)

        shuffle_idx = rng.permutation(len(train_labels))
        train_features = train_features[shuffle_idx]
        train_labels = train_labels[shuffle_idx]

        print(f"\nAfter augmentation: {len(train_labels):,} train windows")
    else:
        print("No classes needed augmentation.")
else:
    print("Augmentation disabled or no training data.")

#### 2. Helper: Evaluate & Log

In [0]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

mlflow.set_experiment(baselines_experiment_name)

def evaluate_and_log(model_name, test_preds, test_labels, run_id=None, test_probs=None,
                     symbols=None, dates=None):
    """Evaluate predictions and print results. Saves to Delta with symbol/date for backtesting."""
    f1_macro = f1_score(test_labels, test_preds, average="macro", zero_division=0)
    f1_weighted = f1_score(test_labels, test_preds, average="weighted", zero_division=0)
    acc = accuracy_score(test_labels, test_preds)

    print(f"Model: {model_name}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Macro F1:    {f1_macro:.4f}")
    print(f"Weighted F1: {f1_weighted:.4f}")
    print(classification_report(
        test_labels, test_preds,
        target_names=[phase_names[i] for i in range(number_of_phases)],
        zero_division=0,
    ))

    # Save predictions to Delta (with symbol/date for downstream backtesting)
    if test_probs is None:
        test_probs_list = [[0.0]*number_of_phases for _ in range(len(test_preds))]
    else:
        test_probs_list = [list(map(float, p)) for p in test_probs]

    pred_rows = [
        (sym, dt, model_name, run_id or "", int(pred), int(label),
         float(f1_macro), float(f1_weighted), float(acc), probs)
        for sym, dt, pred, label, probs in zip(symbols, dates, test_preds, test_labels, test_probs_list)
    ]
    schema = "symbol STRING, date DATE, model_name STRING, run_id STRING, predicted INT, actual INT, macro_f1 DOUBLE, weighted_f1 DOUBLE, accuracy DOUBLE, probabilities ARRAY<DOUBLE>"

    spark.createDataFrame(pred_rows, schema=schema).write.mode("overwrite").saveAsTable(
        f"{catalog}.{gold_schema}.{model_name}_predictions"
    )

    return f1_macro, f1_weighted, acc

#### 3. Baseline 1: Random

In [0]:
with mlflow.start_run(run_name="baseline_random"):
    np.random.seed(42)
    random_preds = np.random.randint(0, number_of_phases, size=len(test_labels))

    f1m, f1w, acc = evaluate_and_log("random", random_preds, test_labels,
                                     symbols=test_symbols, dates=test_dates)
    mlflow.log_metrics({"test_f1_macro": f1m, "test_f1_weighted": f1w, "test_accuracy": acc})
    mlflow.log_param("model_type", "random")

#### 4. Baseline 2: Majority Class

In [0]:
with mlflow.start_run(run_name="baseline_majority"):
    counter = Counter(train_labels)
    majority_class = counter.most_common(1)[0][0]
    majority_preds = np.full(len(test_labels), majority_class)

    print(f"Majority class: {phase_names[majority_class]} ({counter[majority_class]/len(train_labels)*100:.1f}%)")
    f1m, f1w, acc = evaluate_and_log("majority", majority_preds, test_labels,
                                     symbols=test_symbols, dates=test_dates)
    mlflow.log_metrics({"test_f1_macro": f1m, "test_f1_weighted": f1w, "test_accuracy": acc})
    mlflow.log_params({"model_type": "majority", "majority_class": phase_names[majority_class]})

#### 5. Baseline 3: XGBoost

In [0]:
def window_stats(X):
    return np.concatenate([
        X.mean(axis=1),   # level
        X.std(axis=1),    # volatility
        X.min(axis=1),    # trough 
        X.max(axis=1),    # peak 
        X[:, -1, :],      # last bar 
    ], axis=1).astype(np.float32)

with mlflow.start_run(run_name="baseline_xgboost") as run:
    xgb_config = XGBoostConfig()

    #(N, 5*F).
    X_train = window_stats(train_features)
    X_val = window_stats(val_features)
    X_test = window_stats(test_features)

    print(f"XGBoost input shape: {X_train.shape}  (= N, 5 × {num_features} features)")

    dtrain = xgb.DMatrix(X_train, label=train_labels)
    dval = xgb.DMatrix(X_val, label=val_labels)
    del X_train, X_val

    dtest = xgb.DMatrix(X_test, label=test_labels)
    del X_test

    params = {
        "objective": xgb_config.objective,
        "num_class": xgb_config.num_class,
        "eval_metric": xgb_config.eval_metric,
        "max_depth": xgb_config.max_depth,
        "learning_rate": xgb_config.learning_rate,
        "subsample": xgb_config.subsample,
        "colsample_bytree": xgb_config.colsample_bytree,
        "tree_method": "hist",
        "seed": 42,
    }

    bst = xgb.train(
        params, dtrain,
        num_boost_round=xgb_config.n_estimators,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=xgb_config.early_stopping_rounds,
        verbose_eval=50,
    )
    del dtrain, dval

    xgb_probs = bst.predict(dtest)
    xgb_preds = xgb_probs.argmax(axis=1)
    del dtest

    f1m, f1w, acc = evaluate_and_log("xgboost", xgb_preds, test_labels, run.info.run_id, xgb_probs,
                                     symbols=test_symbols, dates=test_dates)

    mlflow.log_metrics({"test_f1_macro": f1m, "test_f1_weighted": f1w, "test_accuracy": acc})
    mlflow.log_params({k: v for k, v in params.items() if k != "objective"})
    mlflow.log_param("input_type", "window_stats_5")
    mlflow.xgboost.log_model(bst, artifact_path="model")


#### 6. Baseline 4: Plain LSTM

In [0]:
class PlainLSTM(nn.Module):
    """BiLSTM + MLP classifier (no CNN, no attention)."""

    def __init__(self, config: LSTMBaselineConfig):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config.num_features,
            hidden_size=config.lstm_hidden,
            num_layers=config.lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=config.lstm_dropout if config.lstm_layers > 1 else 0,
        )
        lstm_out = config.lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out, config.mlp_hidden),
            nn.BatchNorm1d(config.mlp_hidden),
            nn.ReLU(),
            nn.Dropout(config.mlp_dropout),
            nn.Linear(config.mlp_hidden, config.num_classes),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.classifier(last)

In [0]:
class SimpleDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.from_numpy(features)
        self.labels = torch.from_numpy(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [0]:
for hidden_dim in [64, 128]:
    variant_name = f"lstm_h{hidden_dim}"
    print(f"Plain LSTM variant: lstm_hidden={hidden_dim}")

    with mlflow.start_run(run_name=f"baseline_{variant_name}") as run:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        lstm_config = LSTMBaselineConfig()
        lstm_config.num_features = num_features
        lstm_config.lstm_hidden = hidden_dim

        model = PlainLSTM(lstm_config).to(device)
        print(f"LSTM parameters: {sum(p.numel() for p in model.parameters()):,}")

        train_loader = DataLoader(SimpleDataset(train_features, train_labels),
                                  batch_size=lstm_config.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(SimpleDataset(val_features, val_labels),
                                batch_size=lstm_config.batch_size, shuffle=False)
        test_loader = DataLoader(SimpleDataset(test_features, test_labels),
                                 batch_size=lstm_config.batch_size, shuffle=False)

        # Class weights (capped to avoid over-predicting rare classes)
        class_counts = np.bincount(train_labels, minlength=number_of_phases)
        weights = len(train_labels) / (number_of_phases * class_counts.clip(min=1))
        weights = np.clip(weights, 1.0, lstm_config.max_class_weight)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(weights, dtype=torch.float32).to(device),
            label_smoothing=lstm_config.label_smoothing,
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=lstm_config.learning_rate, weight_decay=lstm_config.weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

        best_val_f1 = 0.0
        patience = 0
        best_state = None

        for epoch in range(1, lstm_config.max_epochs + 1):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), lstm_config.grad_clip_norm)
                optimizer.step()

            # Validate
            model.eval()
            val_preds = []
            with torch.no_grad():
                for bx, by in val_loader:
                    val_preds.extend(model(bx.to(device)).argmax(dim=1).cpu().numpy())
            val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Val F1 = {val_f1:.4f}")

            mlflow.log_metric("val_f1", val_f1, step=epoch)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= lstm_config.early_stop_patience:
                    print(f"Early stop at epoch {epoch}")
                    break

        # Restore best and evaluate
        if best_state:
            model.load_state_dict(best_state)
            model.to(device)

        model.eval()
        lstm_preds, lstm_probs_list = [], []
        with torch.no_grad():
            for bx, by in test_loader:
                logits = model(bx.to(device))
                probs = Fn.softmax(logits, dim=1)
                lstm_preds.extend(logits.argmax(dim=1).cpu().numpy())
                lstm_probs_list.extend(probs.cpu().numpy())

        f1m, f1w, acc = evaluate_and_log(variant_name, lstm_preds, test_labels, run.info.run_id, lstm_probs_list,
                                         symbols=test_symbols, dates=test_dates)

        mlflow.log_metrics({"test_f1_macro": f1m, "test_f1_weighted": f1w, "test_accuracy": acc})
        mlflow.log_params({
            "lstm_hidden": lstm_config.lstm_hidden,
            "lstm_layers": lstm_config.lstm_layers,
            "variant": variant_name,
        })
        from mlflow.models import infer_signature
        test_ds = SimpleDataset(test_features, test_labels)
        sample_input = test_ds.features[:2].numpy()
        model.eval()
        with torch.no_grad():
            sample_output = model(test_ds.features[:2].to(device)).cpu().numpy()
        signature = infer_signature(sample_input, sample_output)
        mlflow.pytorch.log_model(model, artifact_path="model", signature=signature)

#### 7. Baseline 4: Plain CNN

In [0]:
class PlainCNN(nn.Module):
    """1D-CNN classifier"""

    def __init__(self, config: CNNBaselineConfig):
        super().__init__()
        c1, c2, c3 = config.conv1_channels, config.conv2_channels, config.conv3_channels
        k1, k2, k3 = config.conv1_kernel, config.conv2_kernel, config.conv3_kernel

        self.block1 = nn.Sequential(
            nn.Conv1d(config.num_features, c1, kernel_size=k1, padding=k1 // 2),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.Conv1d(c1, c1, kernel_size=k1, padding=k1 // 2),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.MaxPool1d(config.pool_size),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(c1, c2, kernel_size=k2, padding=k2 // 2),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.Conv1d(c2, c2, kernel_size=k2, padding=k2 // 2),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.MaxPool1d(config.pool_size),
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(c2, c3, kernel_size=k3, padding=k3 // 2),
            nn.BatchNorm1d(c3),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(config.mlp_dropout),
            nn.Linear(c3, config.mlp_hidden),
            nn.BatchNorm1d(config.mlp_hidden),
            nn.ReLU(),
            nn.Dropout(config.mlp_dropout),
            nn.Linear(config.mlp_hidden, config.num_classes),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        return self.classifier(x)

In [0]:
with mlflow.start_run(run_name="baseline_cnn") as run:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cnn_config = CNNBaselineConfig()
    cnn_config.num_features = num_features

    model = PlainCNN(cnn_config).to(device)
    print(f"CNN parameters: {sum(p.numel() for p in model.parameters()):,}")

    train_loader = DataLoader(SimpleDataset(train_features, train_labels),
                              batch_size=cnn_config.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(SimpleDataset(val_features, val_labels),
                            batch_size=cnn_config.batch_size, shuffle=False)
    test_loader = DataLoader(SimpleDataset(test_features, test_labels),
                             batch_size=cnn_config.batch_size, shuffle=False)

    class_counts = np.bincount(train_labels, minlength=number_of_phases)
    weights = len(train_labels) / (number_of_phases * class_counts.clip(min=1))
    weights = np.clip(weights, 1.0, cnn_config.max_class_weight)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(device),
        label_smoothing=cnn_config.label_smoothing,
    )
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cnn_config.learning_rate,
                                  weight_decay=cnn_config.weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1 = 0.0
    patience = 0
    best_state = None

    for epoch in range(1, cnn_config.max_epochs + 1):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cnn_config.grad_clip_norm)
            optimizer.step()

        model.eval()
        val_preds = []
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                val_loss += criterion(logits, by).item() * bx.size(0)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)

        scheduler.step(metrics=val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Val Loss = {val_loss:.4f}, Val F1 = {val_f1:.4f}, LR = {current_lr:.2e}")

        mlflow.log_metrics({"val_loss": val_loss, "val_f1": val_f1, "lr": current_lr}, step=epoch)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= cnn_config.early_stop_patience:
                print(f"Early stop at epoch {epoch}")
                break

    if best_state:
        model.load_state_dict(best_state)
        model.to(device)

    model.eval()
    cnn_preds, cnn_probs_list = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            logits = model(bx.to(device))
            probs = Fn.softmax(logits, dim=1)
            cnn_preds.extend(logits.argmax(dim=1).cpu().numpy())
            cnn_probs_list.extend(probs.cpu().numpy())

    f1m, f1w, acc = evaluate_and_log("cnn", cnn_preds, test_labels, run.info.run_id, cnn_probs_list,
                                     symbols=test_symbols, dates=test_dates)

    mlflow.log_metrics({"test_f1_macro": f1m, "test_f1_weighted": f1w, "test_accuracy": acc})
    mlflow.log_params({
        "conv_channels": f"{cnn_config.conv1_channels}-{cnn_config.conv2_channels}-{cnn_config.conv3_channels}",
        "conv_kernels": f"{cnn_config.conv1_kernel}-{cnn_config.conv2_kernel}-{cnn_config.conv3_kernel}",
        "pool_size": cnn_config.pool_size,
        "mlp_hidden": cnn_config.mlp_hidden,
        "variant": "cnn",
    })
    from mlflow.models import infer_signature
    test_ds = SimpleDataset(test_features, test_labels)
    sample_input = test_ds.features[:2].numpy()
    model.eval()
    with torch.no_grad():
        sample_output = model(test_ds.features[:2].to(device)).cpu().numpy()
    signature = infer_signature(sample_input, sample_output)
    mlflow.pytorch.log_model(model, artifact_path="model", signature=signature)

#### 8. Summary comparison

In [0]:
print("Baseline comparision sumaary")
print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10} {'Weighted F1':>12}")

for model_name in model_names:
    try:
        tbl = f"{catalog}.{gold_schema}.{model_name}_predictions"
        row = spark.table(tbl).select("macro_f1", "weighted_f1", "accuracy").limit(1).collect()[0]
        print(f"{model_name:<20} {row['accuracy']:>10.4f} {row['macro_f1']:>10.4f} {row['weighted_f1']:>12.4f}")
    except Exception:
        pass